# 00 合成データを見る — 「正常」と「異常」を目で確かめる

> 🔰 **やさしい導入**（この回のひとこと）
>
> - **ひとことで言うと**: 練習用の発電量データを作り、「正常な日」と「異常な日」が見た目でどう違うかを確かめる回です。
> - **たとえ話**: 模擬試験の問題を自分で作り、答え（異常の場所）を手元に持っておくようなもの。
> - **先に読むと分かりやすい**: [00_basics.ipynb](00_basics.ipynb) の「2-1（時系列とふつう）・2-6（合成データと注入）」。用語が分からなくなったら、同じNotebookの「用語集」へ。
> - **難しければ**: 各コードの後ろにある「**出力の見方**」の文章だけ読めば、結論は追えます。

## ゴール
- このプロジェクトで使う**合成の日次発電量データ**の中身を理解する。
- 「発電量が少ない」ことと「異常」が**違う**ことを、データで確かめる。
- 4種類の異常（停電・劣化・影・固着）が、どんな形で混ざっているかを見る。

## やること
1. データを読み込んで表を見る
2. 発電量の時系列を描く
3. 天気と発電量の関係を見る（少ない＝異常ではない）
4. 異常の種類ごとに、正常だった場合の値と比べる

## キーワード
- **異常検知**: 「いつもと違う」ものを見つけること。
- **正常データ**: 異常を含まない、普段どおりのデータ。異常検知では、まずこれだけで「普通」を学ぶ。
- **共変量**: 発電量を説明する別の情報（ここでは日射量・日照率・天気）。
- **注入**: 合成データに、わざと異常を混ぜること。混ぜた場所が分かるので、**答え合わせ**ができる。
- **教師なし**: 「これが異常」という正解ラベルを使わずに学ぶこと。現実の異常検知は、ほぼこれ。

## なぜ合成データなのか
現実の発電量には「異常の正解」がほとんど付いていません。検知器の良し悪しを測るには正解が必要なので、
**正常な系列を作り、異常を自分で混ぜる**ことで、答えのあるデータを用意します。

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
from anomaly_synth import make_split

plt.rcParams["font.family"] = ["Hiragino Sans", "AppleGothic", "DejaVu Sans"]
pd.set_option("display.width", 140)

train = make_split("train")   # 正常のみ（学習用）
val = make_split("val")       # 異常あり（しきい値を決める用）
print(len(train), len(val))
val.head()

**何をしているか**: 学習用（train）と、しきい値を決める用（val）のデータを作って、先頭を表示します。
データは乱数で作りますが、シードが固定なので**何度実行しても同じ**になります。

**出力の見方**（表の列）:
- `weather_class`: 天気（0=晴、1=曇、2=雨）
- `sunshine_ratio`: 日照率（0〜1）、`gsr`: 日射量（MJ/m²）。発電量を説明する共変量です。
- `gen_kwh`: **観測された発電量**。検知器が見るのはこれ。
- `gen_clean_kwh`: 異常を混ぜる**前**の発電量。答え合わせ用で、**検知器には渡してはいけません**。
- `is_anomaly` / `anomaly_type` / `anomaly_id`: 答え（異常日か、種類、何番目の異常か）。これも検知器には渡しません。

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(12, 6))
ax[0].plot(train["date"], train["gen_kwh"], lw=0.8, color="gray")
ax[0].set(title="train（正常のみ）の発電量", ylabel="kWh")

colors = {"outage": "red", "degrade": "orange", "shade": "purple", "stuck": "green"}
ax[1].plot(val["date"], val["gen_kwh"], lw=0.8, color="gray")
for kind, c in colors.items():
    d = val[val["anomaly_type"] == kind]
    ax[1].scatter(d["date"], d["gen_kwh"], s=14, color=c, label=kind, zorder=3)
ax[1].set(title="val（異常あり）。色の点が異常日", ylabel="kWh")
ax[1].legend(ncol=4)
fig.tight_layout()

**何をしているか**: 上が正常のみ、下が異常を混ぜたデータです。異常日は色付きの点で示します。

**出力の見方**:
- 夏に多く、冬に少ない**季節の波**があり、その上で日々大きく上下しています（天気のせい）。
- 下の図の色付き点を**色を消して**眺めてみてください。灰色の線だけで、異常がどこか分かるでしょうか。
  多くは「ただの曇りの日」と見分けがつきません。**ここが異常検知の難しさ**です。

In [ ]:
labels = {0: "晴", 1: "曇", 2: "雨"}
g = train.groupby("weather_class")["gen_kwh"].agg(["count", "mean", "min", "max"]).round(1)
g.index = [labels[i] for i in g.index]
g

**何をしているか**: 正常データ（train）で、天気ごとの発電量を集計します。

**出力の見方**: 雨の日の平均は晴れの日より大幅に小さいはずです。
でも、雨の日に発電量が少ないのは**正常**です。もし「小さい値＝異常」というルールにすると、雨の日をすべて異常と誤って警報を出してしまいます。
**天気で説明できない低下だけが異常**、というのがこのプロジェクトの考え方です。

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].scatter(train["gsr"], train["gen_kwh"], s=6, alpha=0.5)
ax[0].set(title="正常: 日射量と発電量はほぼ比例", xlabel="日射量 gsr (MJ/m²)", ylabel="発電量 kWh")
ok, bad = val[val["is_anomaly"] == 0], val[val["is_anomaly"] == 1]
ax[1].scatter(ok["gsr"], ok["gen_kwh"], s=6, alpha=0.4, color="gray", label="正常日")
ax[1].scatter(bad["gsr"], bad["gen_kwh"], s=10, color="red", label="異常日")
ax[1].set(title="val: 異常日は線から外れる", xlabel="日射量 gsr (MJ/m²)", ylabel="発電量 kWh")
ax[1].legend()
fig.tight_layout()

**何をしているか**: 横軸を日射量、縦軸を発電量にした散布図です。

**出力の見方**:
- 左（正常）: 点がほぼ**1本の線**に乗っています。日射量が分かれば、発電量はかなり当てられます。
  これが「**天気を考慮した期待値**」の元になります（Phase2）。
- 右（val）: 赤い点（異常日）のうち、停電と影の日は線のかなり**下側**に外れています。時系列では見分けにくかった異常が、日射量との関係なら見つけやすくなります。
- 一方、劣化の日の多くは線の**すぐ近く**にあり、固着の日は線の下にも上にも散らばっています。**弱い異常や、値の大きさに特徴のない異常は、正常のばらつきに埋もれる**のが分かります。

In [ ]:
rows = []
for kind in ["outage", "degrade", "shade", "stuck"]:
    d = val[val["anomaly_type"] == kind]
    ratio = (d["gen_kwh"] / d["gen_clean_kwh"]).round(2)
    rows.append(dict(種類=kind, 異常の数=d["anomaly_id"].nunique(), 異常日数=len(d),
                     比_最小=ratio.min(), 比_平均=round(ratio.mean(), 2), 比_最大=ratio.max()))
pd.DataFrame(rows)

**何をしているか**: 異常の種類ごとに、「観測値 ÷ 異常がなかった場合の値」（比）を集計します（答え合わせ列を使うのは、この確認のためだけです）。

**出力の見方**:
- `outage`（停電）: 比が0〜0.1。ほぼ発電ゼロ。**易しい**異常です。
- `shade`（影）: 比が0.6〜0.8。2〜4割の低下。曇りの日のばらつきに紛れやすい。
- `degrade`（劣化）: 比は0.9〜0.98程度（この系列）。低下は1割に届かず、**徐々に**悪化するので、正常のばらつき（約3%の発電ノイズ、曇りの日の大きな変動）に紛れます。
- `stuck`（固着）: 比は0.18〜8.29とまちまち（天気が変わっても値が変わらないため）。**値が動かない**ことが特徴で、値の大きさでは判断できません。

In [ ]:
d = val[val["anomaly_type"] == "stuck"]
first = d.index[0]
val.loc[first - 2 : first + 5, ["date", "gsr", "weather_class", "gen_kwh", "anomaly_type"]]

**何をしているか**: 固着（stuck）が始まる前後の数日を表示します。

**出力の見方**: `gen_kwh` が数日間**同じ値のまま**です。一方、`gsr`（日射量）と `weather_class` は日々変わっています。
つまり「天気は変わっているのに発電量は変わらない」ことが異常のサインです。**値の大きさだけを見る手法では見つけられません**。

## 試してみよう（演習）
1. **異常日の割合を確認する**: 次のセルで `val["is_anomaly"].mean()` を実行してください。
   現実の異常日は1〜3%程度ですが、このデータは評価の統計を取るため、1〜2割と**多め**にしてあります。この違いが評価にどう影響しそうか考えてみましょう。
2. **別の系列を見る**: 最初のコードセルの `make_split("val")` を `make_split("val", 1)` に変えて実行し直してください。
   異常の位置が変わります。同じ手法でも、系列によって検知できたりできなかったりします（だから複数の系列で評価します）。
3. **異常を減らす**: `make_split("val", 0, {"outage": 2})` のように異常の数を指定すると、停電2回だけのデータになります。他の種類を0にして、図の見え方を比べてみましょう。

## まとめ
- 発電量が少ないだけでは異常ではない。**天気（日射量）で説明できない**ずれが異常。
- 異常の種類によって、見つけやすさが大きく違う（停電は易しく、劣化・固着は難しい）。
- 答え（`gen_clean_kwh`・`is_anomaly` など）は、**評価の答え合わせにだけ**使う。検知器には渡さない。

次は Phase1 で、まず**AIを使わない**簡単な方法（しきい値・移動平均）で異常を探し、どこで失敗するかを見ます。